# Donut Fine-Tuning for SurveyPlan AI
This notebook contains the complete pipeline for fine-tuning the `naver-clova-ix/donut-base` model to extract structured data from survey plans. It includes parsing geometry, tables, and titles using custom schemas.

## 1. Setup & Environment
Pull the latest changes from the GitHub repository and install required dependencies. We also mount Google Drive to access the dataset.

In [ ]:
import os
# Check if the repository already exists to avoid double cloning
if not os.path.exists('survey-plan-AI'):
    !git clone https://github.com/14jadon14/survey-plan-AI.git
else:
    !git -C survey-plan-AI pull origin main

%cd survey-plan-AI

# scikit-learn is pre-installed on Colab; do NOT use 'sklearn' (broken shim package)
!pip install -q transformers datasets pytorch-lightning sentencepiece torchvision accelerate editdistance scikit-learn

from google.colab import drive
drive.mount('/content/drive')


## 2. Load Dataset
We load the images and `metadata.jsonl` from the Google Drive directory. The dataset is split into training and validation sets.

In [ ]:
from datasets import load_dataset
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
import json
import os

# Path where the metadata.jsonl and crops are stored
DATA_DIR = Path('/content/drive/MyDrive/SurveyPlan AI/runs/donut_tuning')
os.makedirs(DATA_DIR, exist_ok=True)

try:
    # Pre-process metadata.jsonl to remove missing images before loading
    metadata_path = DATA_DIR / 'metadata.jsonl'
    if metadata_path.exists():
        valid_entries = []
        missing_count = 0
        with open(metadata_path, 'r', encoding='utf-8') as f:
            for line in f:
                entry = json.loads(line)
                img_path = DATA_DIR / entry['file_name']
                if img_path.exists():
                    valid_entries.append(line)
                else:
                    missing_count += 1
                    print(f"Skipping missing file: {entry['file_name']}")
        
        if missing_count > 0:
            print(f"Removed {missing_count} missing entries from metadata.jsonl")
            with open(metadata_path, 'w', encoding='utf-8') as f:
                f.writelines(valid_entries)

    # Load dataset using HuggingFace's imagefolder script
    dataset = load_dataset("imagefolder", data_dir=str(DATA_DIR), split="train", download_mode="force_redownload")

    # --- STRATIFIED SPLIT ---
    # Guarantees every label has at least 1 sample in the validation set.
    # This prevents missing per-label CER stats during validation.
    all_labels = []
    for item in dataset:
        gt = item['ground_truth']
        if isinstance(gt, str): gt = json.loads(gt)
        # Use the 'label' field from the metadata for stratification
        all_labels.append(item.get('label', 'unknown'))

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
    train_idx, val_idx = next(sss.split(range(len(dataset)), all_labels))

    train_dataset = dataset.select(train_idx.tolist())
    val_dataset = dataset.select(val_idx.tolist())

    # Print split summary per label
    from collections import Counter
    train_labels = Counter(all_labels[i] for i in train_idx)
    val_labels = Counter(all_labels[i] for i in val_idx)
    print(f"\nSTRATIFIED SPLIT: {len(train_dataset)} train, {len(val_dataset)} val")
    print(f"{'Label':<16} {'Train':>6} {'Val':>5}")
    print("-" * 30)
    for label in sorted(set(all_labels)):
        print(f"{label:<16} {train_labels.get(label,0):>6} {val_labels.get(label,0):>5}")

except Exception as e:
    print(f"ERROR LOADING DATASET: {e}")


## 2.5 Hyperparameters
All tunable hyper-parameters parameters are exposed here. Adjust these values depending on your GPU VRAM, target accuracy, and total dataset size without needing to edit the training logic below.

In [ ]:
# === TUNABLE HYPERPARAMETERS ===
MAX_EPOCHS = 30
BATCH_SIZE = 2
NUM_WORKERS = 2
LEARNING_RATE = 2e-5
MAX_LENGTH = 512
IMAGE_RESOLUTION = {"height": 1280, "width": 960}
# ===============================

## 3. Configure Processor & Tokenizer
Donut requires a high-resolution input for readable text (1280x960). We also add our custom schema tokens to the tokenizer's vocabulary to ensure the model recognizes our specific JSON keys.

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from tqdm import tqdm
import json

model_id = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(model_id)
processor.image_processor.size = IMAGE_RESOLUTION

# Dynamically extract schema tokens from the dataset
task_start_tokens = set()
schema_tokens = set()

def extract_tokens(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            schema_tokens.add(f"<{k}>")
            extract_tokens(v)
    elif isinstance(obj, list):
        for item in obj:
            extract_tokens(item)

print("Analyzing dataset for tokens...")
for item in tqdm(train_dataset):
    try:
        gt = item['ground_truth']
        if isinstance(gt, str): gt = json.loads(gt)
        gt_parse = gt.get('gt_parse', gt)
        
        if isinstance(gt_parse, dict) and len(gt_parse) == 1:
            root_key = list(gt_parse.keys())[0]
            task_start_tokens.add(f"<s_{root_key}>")
            extract_tokens(gt_parse[root_key])
    except FileNotFoundError:
        # Skip missing files if they weren't caught by pre-processing
        continue
    except Exception as e:
        print(f"Warning: Skipping an item due to error: {e}")

task_start_tokens = sorted(list(task_start_tokens))
schema_tokens = sorted(list(schema_tokens))

new_tokens = task_start_tokens + schema_tokens
for token in schema_tokens:
    new_tokens.append(token.replace("<", "</"))

processor.tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
print(f"\nDynamically discovered {len(task_start_tokens)} task tokens and {len(schema_tokens)} schema tokens!")

## 4. Model Initialization
We load the base Donut model and adjust its token embeddings to account for the newly added schema tokens.

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(model_id)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Dynamically set the decoder start token ID to prevent hallucination stuttering
if task_start_tokens:
    general_token_id = processor.tokenizer.convert_tokens_to_ids(task_start_tokens[0])
else:
    general_token_id = processor.tokenizer.convert_tokens_to_ids("<s_general>")

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = general_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
print(f"Model successfully initialized and resized. Decoder Start Token ID: {general_token_id}")


## 5. Dataset Loader Definition
Here we define the PyTorch Dataset. A custom `json2token` helper recursively converts the JSON ground truth into the sequence string format Donut natively understands (e.g. `<s_general><text>...</text></s_general>`).

In [ ]:
from torch.utils.data import Dataset, DataLoader
import json
import torch

def json2token(obj, sort_json_key: bool = True):
    """
    Converts a JSON object (dict) into a token sequence for training.
    Values must be plain strings (no embedded XML tags).
    """
    if isinstance(obj, dict):
        output = ""
        keys = sorted(obj.keys()) if sort_json_key else obj.keys()
        for k in keys:
            output += f"<{k}>"
            output += json2token(obj[k], sort_json_key)
            output += f"</{k}>"
        return output
    elif isinstance(obj, list):
        return "".join([json2token(item, sort_json_key) for item in obj])
    else:
        return str(obj)

class DonutDataset(Dataset):
    def __init__(self, dataset, processor, max_length=MAX_LENGTH, split="train", task_prompt="<s_general>"):
        super().__init__()
        self.dataset = dataset
        self.processor = processor
        self.max_length = max_length
        self.split = split
        self.task_prompt = task_prompt

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        
        gt = item['ground_truth']
        if isinstance(gt, str): gt = json.loads(gt)
        gt_parse = gt.get('gt_parse', gt)
        
        # --- PHASE 1: Extract the label_type from the metadata ---
        # 'label' is the raw leaf key set by generate_metadata.py (e.g. 'notes', 'coord table')
        label_type = item.get('label', 'unknown')
        
        # Automatically select prompt based on root key
        prompt = self.task_prompt
        if prompt is None:
            prompt = task_start_tokens[0] if task_start_tokens else "<s_general>"
        if isinstance(gt_parse, dict) and len(gt_parse) == 1:
            root_key = list(gt_parse.keys())[0]
            if f"<s_{root_key}>" in task_start_tokens:
                prompt = f"<s_{root_key}>"
                # Unwrap the redundant root key to simplify sequence generation
                gt_parse = gt_parse[root_key]
                
        sequence = prompt + json2token(gt_parse) + self.processor.tokenizer.eos_token
        
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            sequence, add_special_tokens=False, 
            max_length=self.max_length, padding="max_length", 
            truncation=True, return_tensors="pt"
        )["input_ids"].squeeze()
        
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        # label_type is returned as a string for per-category CER grouping in validation
        return {"pixel_values": pixel_values, "labels": labels, "label_type": label_type}

# Optimize Data Loading for final dataset: increase batch_size and num_workers as needed
train_ds = DonutDataset(train_dataset, processor)
val_ds = DonutDataset(val_dataset, processor, split="val")
# TWEAK THESE HYPERPARAMETERS FOR FASTER TRAINING
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


## 6. PyTorch Lightning Module
We wrap the model in a PyTorch Lightning module. This handles the training loop and implements a `validation_step` that computes the **Character Error Rate (CER)** metric, logging it to the progress bar. We aim to keep CER below 0.05.

In [ ]:
# --- DONUT MODEL MODULE: Advanced Stability Metrics ---
import pytorch_lightning as pl
import editdistance
from collections import defaultdict

# ------------------------------------------------------------------
# CER CONVENTION:
#   raw_cer = editdistance(pred, gt) / max(len(gt), 1)  # ratio 0.0 - inf
#   cer_pct = min(raw_cer, 1.0) * 100                   # PERCENTAGE 0 - 100 %
# Capping at 100% prevents inflated values from long garbage outputs.
# Every metric in this class is a PERCENTAGE (0-100%).
# ------------------------------------------------------------------

# All categories expected in the full dataset.
ALL_EXPECTED_LABELS = {
    "adj lot", "area", "azimuth", "coord table", "curve data",
    "distance", "lot number", "notes", "plan title", "street", "text",
}

class DonutModelPLModule(pl.LightningModule):
    def __init__(self, model, processor, learning_rate=LEARNING_RATE):
        super().__init__()
        self.model = model
        self.processor = processor
        self.learning_rate = learning_rate
        self.best_cer = float('inf')        # best cer_pct across all epochs
        self.epoch_cer_history = []          # mean cer_pct per completed epoch
        self.validation_step_outputs = []    # per-sample {cer, label} for current epoch
        self._warned_missing_labels = False  # print missing-label warning only once

    def training_step(self, batch, batch_idx):
        outputs = self.model(pixel_values=batch["pixel_values"], labels=batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        labels = batch["labels"].clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id

        label_types = batch.get("label_type", ["unknown"] * pixel_values.size(0))

        prompt_ids = labels[:, :1]
        outputs = self.model.generate(
            pixel_values,
            decoder_input_ids=prompt_ids,
            max_length=self.model.decoder.config.max_position_embeddings,
            pad_token_id=self.processor.tokenizer.pad_token_id,
            eos_token_id=self.processor.tokenizer.eos_token_id,
            use_cache=True, return_dict_in_generate=True,
        )

        preds = self.processor.tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)
        gts   = self.processor.tokenizer.batch_decode(labels,           skip_special_tokens=True)

        batch_cers = []
        for pred, gt, label in zip(preds, gts, label_types):
            dist    = editdistance.eval(pred.strip(), gt.strip())
            raw_cer = dist / max(len(gt.strip()), 1)   # ratio, can exceed 1.0
            cer_pct = min(raw_cer, 1.0) * 100           # PERCENTAGE 0-100, capped
            batch_cers.append(cer_pct)
            self.validation_step_outputs.append({"cer": cer_pct, "label": label})

        batch_avg_cer = sum(batch_cers) / len(batch_cers)
        self.log("val_cer", batch_avg_cer, prog_bar=True)
        return batch_avg_cer

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking or not self.validation_step_outputs:
            self.validation_step_outputs.clear()
            return

        all_cers = [o['cer'] for o in self.validation_step_outputs]
        epoch_mean_cer = sum(all_cers) / len(all_cers)  # percentage 0-100

        # 1. Best-to-Date
        if epoch_mean_cer < self.best_cer:
            self.best_cer = epoch_mean_cer
        self.log("best_cer_so_far", self.best_cer, prog_bar=True)

        # 2. Rolling 3-Epoch
        self.epoch_cer_history.append(epoch_mean_cer)
        window = self.epoch_cer_history[-3:]
        rolling_3_cer = sum(window) / len(window)
        self.log("rolling_3_cer", rolling_3_cer, prog_bar=True)

        # 3. Cumulative Average
        cumulative_avg_cer = sum(self.epoch_cer_history) / len(self.epoch_cer_history)
        self.log("cumulative_avg_cer", cumulative_avg_cer, prog_bar=False)

        # 4. Per-Label CER
        label_cers = defaultdict(list)
        for o in self.validation_step_outputs:
            label_cers[o["label"]].append(o["cer"])

        label_parts = []
        for label, cers in sorted(label_cers.items()):
            avg_pct = sum(cers) / len(cers)
            self.log("val_cer_" + label.lower().replace(" ", "_").replace("-", "_"), avg_pct, prog_bar=False)
            label_parts.append(f"{label}={avg_pct:.1f}%")

        # Console summary (compact)
        print(f"  [Epoch {self.current_epoch}] cer={epoch_mean_cer:.2f}%  rolling_3={rolling_3_cer:.2f}%  best={self.best_cer:.2f}%  cumul={cumulative_avg_cer:.2f}%")
        print(f"  Per-label: {'  '.join(label_parts)}")

        # Missing-label warning: print ONCE, not every epoch
        missing = ALL_EXPECTED_LABELS - set(label_cers.keys())
        if missing and not self._warned_missing_labels:
            self._warned_missing_labels = True
            print(f"  NOTE: Labels absent from val split: {sorted(missing)}. Consider stratified splitting.")

        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

pl_module = DonutModelPLModule(model, processor)


## 7. Start Training
Initialize the PyTorch Lightning Trainer and begin the fine-tuning process. We set `log_every_n_steps` low enough to see frequent updates in the console.

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint

# Save the single best checkpoint by val_cer (lower is better)
checkpoint_callback = ModelCheckpoint(
    monitor="val_cer",
    save_top_k=1,
    mode="min",
    filename="donut-best-{epoch:02d}-{val_cer:.2f}",
    verbose=True,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    precision="16-mixed",
    log_every_n_steps=1,
    callbacks=[checkpoint_callback],
)

print("Training STARTING...")
# Force model back into train mode (in case inference was run earlier)
model.train()
for param in model.parameters():
    param.requires_grad = True

print("Model is now in TRAIN mode and weights are unfrozen.")
trainer.fit(pl_module, train_loader, val_loader)

print("\n=== TRAINING COMPLETE ===")
print(f"Best checkpoint saved at: {checkpoint_callback.best_model_path}")
print(f"Best val_cer achieved:    {checkpoint_callback.best_model_score:.4f}%")


## 8. Export Model
Save the fine-tuned model weights and the tokenizer configurations physically back to Google Drive.

In [ ]:
import os
from pytorch_lightning.callbacks import ModelCheckpoint

EXPORT_DIR = '/content/drive/MyDrive/SurveyPlan AI/Models/Donut_Finetuned'
os.makedirs(EXPORT_DIR, exist_ok=True)

# ── 1. Reload the best checkpoint weights into the Lightning module ───────
best_ckpt_path = trainer.checkpoint_callback.best_model_path

if best_ckpt_path:
    print(f"Loading best checkpoint from: {best_ckpt_path}")
    pl_module = DonutModelPLModule.load_from_checkpoint(
        best_ckpt_path,
        model=model,
        processor=processor,
    )
    # Expose the reloaded HF model for save_pretrained and run_direct_inference
    model = pl_module.model
    print("Best weights reloaded successfully.")
else:
    print("WARNING: No checkpoint path found — saving weights from last epoch.")

# ── 2. Save the HuggingFace model and processor to Drive ─────────────────
print(f"Saving to {EXPORT_DIR}...")
model.save_pretrained(EXPORT_DIR)
processor.save_pretrained(EXPORT_DIR)
print("Success!")


## 9. Inference & Evaluation Test
This cell evaluates the model's performance on exactly one sample from the dataset. It prints the actual JSON alongside the model's generated JSON. Finally, it outputs a strict set of **Statistics** measuring how successfully the model memorized or learned the prompt.

In [ ]:
import editdistance
from PIL import Image
import re
import json
import torch

def custom_token2json(sequence):
    """
    Parses a token sequence produced by this model back into a Python dict.
    Handles the XML-style tags produced by json2token, e.g.:
      <s_general><text>hello</text></s_general>  ->  {'general': {'text': 'hello'}}
    """
    sequence = re.sub(r'^<s_[^>]+>', '', sequence.strip())
    sequence = re.sub(r'</s_[^>]+>$', '', sequence.strip())

    def parse_inner(s):
        result = {}
        while s:
            m = re.match(r'<([^/][^>]*)>', s)
            if not m:
                break
            tag = m.group(1)
            s = s[m.end():]
            depth = 1
            pos = 0
            while pos < len(s) and depth > 0:
                open_m = re.search(f'<{re.escape(tag)}>', s[pos:])
                close_m = re.search(f'</{re.escape(tag)}>', s[pos:])
                if close_m and (not open_m or close_m.start() < open_m.start()):
                    pos += close_m.end()
                    depth -= 1
                elif open_m:
                    pos += open_m.end()
                    depth += 1
                else:
                    pos = len(s)
                    depth = 0
            inner = s[:pos - len(f'</{tag}>')]
            s = s[pos:]
            if re.match(r'\s*<[^/]', inner):
                children = re.findall(r'<([^/][^>]*)>.*?</\1>', inner, re.DOTALL)
                child_tags = list(set(children))
                if len(child_tags) == 1 and inner.count(f'<{child_tags[0]}>') > 1:
                    result[tag] = [parse_inner(m.group(1)) 
                                   for m in re.finditer(f'<{child_tags[0]}>(.*?)</{child_tags[0]}>', inner, re.DOTALL)]
                else:
                    result[tag] = parse_inner(inner)
            else:
                result[tag] = inner.strip()
        return result

    return parse_inner(sequence)

def run_direct_inference(image, task_prompt=None):
    if task_prompt is None:
        task_prompt = task_start_tokens[0] if task_start_tokens else "<s_general>"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    
    pixel_values = processor(image.convert("RGB"), return_tensors="pt").pixel_values
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids
    
    # Greedy decoding is favored for strict OCR extraction
    outputs = model.generate(
        pixel_values.to(device),
        decoder_input_ids=decoder_input_ids.to(device),
        max_length=512,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=1,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )
    
    sequence = processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
    
    try:
        json_res = custom_token2json(sequence)
        task_key = task_prompt.replace('<s_', '').replace('>', '')
        if task_key and task_key not in json_res:
            json_res = {task_key: json_res}
    except Exception as e:
        json_res = {"raw_sequence": sequence, "parse_error": str(e)}
        
    return sequence, json_res

print("Preparing dataset sample for Inference...")
sample_idx = 0 
sample_item = train_dataset[sample_idx]
sample_img = sample_item['image']

gt_raw = sample_item['ground_truth']
if isinstance(gt_raw, str): gt_raw = json.loads(gt_raw)

task_prompt = task_start_tokens[0] if task_start_tokens else "<s_general>"
gt_parse = gt_raw.get('gt_parse', gt_raw)
if isinstance(gt_parse, dict) and len(gt_parse) == 1:
    root_key = list(gt_parse.keys())[0]
    if f"<s_{root_key}>" in task_start_tokens:
        task_prompt = f"<s_{root_key}>"

inner_gt = gt_parse.get(root_key, gt_parse) if 'root_key' in locals() else gt_parse
target_sequence = task_prompt + json2token(inner_gt)

pred_seq, pred_json = run_direct_inference(sample_img, task_prompt)

char_dist = editdistance.eval(pred_seq, target_sequence)
cer_percentage = (char_dist / max(len(target_sequence), 1)) * 100
exact_match_seq = (pred_seq == target_sequence)

def count_exact_matches(gt, pred):
    if not isinstance(gt, dict) or not isinstance(pred, dict): return 0, 1
    matches = 0
    total = 0
    for k, v in gt.items():
        if isinstance(v, dict):
            m, t = count_exact_matches(v, pred.get(k, {}))
            matches += m
            total += t
        elif isinstance(v, list):
            total += len(v)
            if k in pred and isinstance(pred[k], list):
                for i in range(min(len(v), len(pred[k]))):
                    m, t = count_exact_matches(v[i], pred[k][i])
                    if m == t and t > 0: matches += 1
        else:
            total += 1
            if k in pred and str(pred[k]).strip() == str(v).strip():
                matches += 1
    return matches, total

try:
    gt_inner = gt_raw.get('gt_parse', gt_raw)
    field_matches, field_total = count_exact_matches(gt_inner, pred_json)
    field_acc = (field_matches / max(field_total, 1)) * 100
except Exception:
    field_acc = 0.0
    field_matches = 0
    field_total = 1

print("\n" + "="*50)
print("INFERENCE EVALUATION")
print("="*50)
print(f"Task Prompt: {task_prompt}\n")
print("--- EXPECTED (Ground Truth) ---")
print(json.dumps(gt_raw, indent=2))
print("\n--- PREDICTED ---")
print(json.dumps(pred_json, indent=2))

print("\n" + "="*50)
print("STATISTICS")
print("="*50)
print(f"Character Error Rate (CER): {cer_percentage:.2f}%")
print(f"Sequence Exact Match:       {'Yes ✅' if exact_match_seq else 'No ❌'}")
print(f"Field-Level Exact Match:    {field_matches}/{field_total} fields correct ({field_acc:.1f}%)")


## 10. Active Learning Auto-Labeler
Once you have used your YOLO pipeline to save *new* blank crops to your Drive (where `ground_truth` is `{"gt_parse": {"text": ""}}`), you can run this cell! It will load your newly fine-tuned Donut model, predict the correct JSON tags for the new images, and overwrite the blank rows in your `metadata.jsonl` with the AI's best guess. This is the **Data Flywheel**!

In [ ]:
import os
import json
from PIL import Image
from tqdm import tqdm
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

# Ensure model is loaded
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

DATA_DIR = "/content/drive/MyDrive/SurveyPlan AI/runs/donut_tuning"
metadata_path = os.path.join(DATA_DIR, "metadata.jsonl")
backup_path = os.path.join(DATA_DIR, "metadata_backup.jsonl")

if not os.path.exists(metadata_path):
    print(f"File not found: {metadata_path}")
else:
    # Create a quick backup just in case
    !cp "{metadata_path}" "{backup_path}"
    
    # Read current metadata
    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    new_lines = []
    labeled_count = 0
    
    print("Scanning metadata for blank templates...")
    for line in tqdm(lines, desc="Auto-Labeling"):
        record = json.loads(line)
        gt = record.get("ground_truth", "")
        
        # Check if it's the blank template exported by YOLO
        if '{"gt_parse": {"text": ""}}' in gt or '"text": ""' in gt:
            img_path = os.path.join(DATA_DIR, record["file_name"])
            label = record.get("label", "general")
            
            if os.path.exists(img_path):
                # 1. Determine Prompt (Schema root)
                task_prompt = "<s_general>"
                if "curve_data" in label:
                    task_prompt = "<s_lot_geometry>"
                elif "parcel_info" in label or "lot_id" in label:
                    task_prompt = "<s_parcel_info>"
                elif "table" in label:
                    task_prompt = "<s_tabular_data>"
                elif "dist" in label or "az" in label or "rad" in label or "arc" in label:
                    task_prompt = "<s_lot_geometry>"
                    
                # 2. Run Inference
                try:
                    img = Image.open(img_path).convert("RGB")
                    pred_seq, pred_json = run_direct_inference(img, task_prompt)
                    
                    # 3. Format Prediction into Ground Truth string
                    # Assuming pred_json directly resembles {"general": {"text": "123"}}
                    new_gt = {"gt_parse": pred_json}
                    record["ground_truth"] = json.dumps(new_gt, ensure_ascii=False)
                    labeled_count += 1
                except Exception as e:
                    print(f"Failed to infer {img_path}: {e}")
                    
        new_lines.append(json.dumps(record, ensure_ascii=False) + "\n")
        
    # Write back the new annotated metadata
    with open(metadata_path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)
        
    print(f"\n[SUCCESS] Auto-Labeled {labeled_count} new images using the Donut Model!")
    print("Open Google Drive, review the JSON file to fix minor mistakes, and you are ready to train again!")
